# Crop class harmonisation

This notebook compares CROME 2022 labels with UKCEH Land Cover plus: Crops 2022, joins the official CROME LUCODE lookup and applies the crop class crosswalk.

In [ ]:
#Extract the UKCEH archive.
!unzip -q /content/drive/MyDrive/Dissertation/ukceh_east_anglia.zip -d /content/

In [ ]:
#Extract the CROME archive.
!unzip -q /content/drive/MyDrive/Dissertation/crome/2022/crome_2022_selected.zip -d /content/

In [ ]:
#load and inspect UKCEH and CROME classes.
import geopandas as gpd
import pandas as pd
lccm_path = '/content/ukceh_east_anglia/lccm-2022_6395063/lccm-2022_6395063.gpkg'
lccm_gdf = gpd.read_file(lccm_path)
crome_path = '/content/crome_2022_selected/east_anglia/Crop_Map_of_England_2022_Norfolk.geojson'
crome_gdf = gpd.read_file(crome_path)
print('--- LCCM 2022 classes ---')
if 'crop_name' in lccm_gdf.columns:
    display(lccm_gdf['crop_name'].unique())
else:
    display(lccm_gdf.columns.tolist())
    display(lccm_gdf.iloc[:, 0:5].head())
print('\n--- CROME 2022 (Norfolk) classes ---')
if 'luc_name' in crome_gdf.columns:
    display(crome_gdf['luc_name'].unique())
else:
    display(crome_gdf.columns.tolist())
    display(crome_gdf.iloc[:, 0:5].head())

In [ ]:
#Join CROME codes to lookup labels
import pandas as pd
lookup_path = '/content/CROME_LUCODE_LOOKUP(Sheet1).csv'
try:
    lookup_df = pd.read_csv(lookup_path, encoding='cp1252')
except UnicodeDecodeError:
    lookup_df = pd.read_csv(lookup_path, encoding='ISO-8859-1')
print("--- CROME Lookup Table ---")
display(lookup_df.head())
print("\n--- CROME Data Unique IDs ---")
unique_ids = crome_gdf['lucode'].unique() if 'lucode' in crome_gdf.columns else crome_gdf.columns
display(unique_ids)
if 'lucode' in crome_gdf.columns and any(col in lookup_df.columns for col in ['lucode', 'LUCODE']):
    lookup_col = 'lucode' if 'lucode' in lookup_df.columns else 'LUCODE'
    crome_with_names = crome_gdf.merge(lookup_df, left_on='lucode', right_on=lookup_col, how='left')
    print("\n--- Mapped CROME classes ---")
    possible_desc_cols = ['desc', 'Description', 'CROP_NAME', 'Crop']
    crop_name_col = next((c for c in possible_desc_cols if c in crome_with_names.columns), None)
    if crop_name_col:
        display(crome_with_names[crop_name_col].unique())
    else:
        print("Candidate crop-description columns:", crome_with_names.columns.tolist())
else:
    print("\nNo matching lookup columns found.")

In [ ]:
#Compare harmonised class names
crop_desc_col = 'Land Use Description' if 'Land Use Description' in crome_with_names.columns else 'Land Cover Description'
print(f"--- LCCM 2022 classes ({len(lccm_gdf['crop_name'].unique())} classes) ---")
lccm_crops = sorted(lccm_gdf['crop_name'].unique().tolist())
display(lccm_crops)
print(f"\n--- CROME 2022 (Norfolk) classes ({len(crome_with_names[crop_desc_col].dropna().unique())} classes) ---")
crome_crops = sorted(crome_with_names[crop_desc_col].dropna().unique().tolist())
display(crome_crops)
common = set(lccm_crops).intersection(set(crome_crops))
print(f"\nShared class names: {len(common)}")
if common:
    display(list(common))
print("\nUse a crosswalk for unmatched names.")

In [ ]:
#Define the CROME to LCCM(UKCEH) crosswalk.
crome_to_lccm_map = {
    'Winter Wheat': 'Winter wheat',
    'Spring Wheat': 'Spring wheat',
    'Winter Barley': 'Winter barley',
    'Spring Barley': 'Spring barley',
    'Winter Oats': 'Winter oats',
    'Spring Oats': 'Spring oats',
    'Winter Field beans': 'Winter field beans',
    'Spring Field beans': 'Spring field beans',
    'Winter Oilseed': 'Oilseed rape',
    'Maize': 'Maize',
    'Potato': 'Potatoes',
    'Beet': 'Beet (sugar beet / fodder beet)',
    'Grass': 'Grass',
    'Spring Peas': 'Peas',
    'Clover': 'Other crops',
    'Fallow Land': 'Other crops',
    'Lucerne': 'Other crops',
    'Mixed Crop-Group 1': 'Other crops',
    'Onions': 'Other crops',
    'Spring Cabbage': 'Other crops',
    'Spring Linseed ': 'Other crops',
    'Winter Linseed': 'Other crops',
    'Winter Rye': 'Other crops',
    'Winter Triticale': 'Other crops',
    'Trees and Scrubs, short Woody plants, hedgerows': 'Other crops',
    'Perennial Crops and Isolated Trees': 'Other crops',
    'Water': 'Water',
    'Non-vegetated or sparsely-vegetated Land': 'Non-vegetated or sparsely-vegetated Land'
}
crome_with_names['lccm_aligned_name'] = crome_with_names[crop_desc_col].map(crome_to_lccm_map)
print("--- Final CROME class distribution ---")
display(crome_with_names['lccm_aligned_name'].value_counts())
unmapped = crome_with_names[crome_with_names['lccm_aligned_name'].isna()][crop_desc_col].unique()
if len(unmapped) == 0:
    print("\nAll classes mapped.")
else:
    print("\nUnmapped classes:", unmapped)

In [ ]:
#Apply the crosswalk to all CROME files.
import os
import glob
def process_crome_file(file_path, lookup_df, mapping_dict):
    gdf = gpd.read_file(file_path)
    lookup_col = 'lucode' if 'lucode' in lookup_df.columns else 'LUCODE'
    gdf = gdf.merge(lookup_df, left_on='lucode', right_on=lookup_col, how='left')
    desc_col = 'Land Use Description' if 'Land Use Description' in gdf.columns else 'Land Cover Description'
    gdf['lccm_aligned_name'] = gdf[desc_col].map(mapping_dict)
    return gdf
all_crome_files = glob.glob('/content/crome_2022_selected/**/*.geojson', recursive=True)
processed_gdfs = {}
print(f"Processing {len(all_crome_files)} areas...")
for file_path in all_crome_files:
    area_name = os.path.basename(file_path).replace('Crop_Map_of_England_2022_', '').replace('.geojson', '')
    try:
        processed_gdfs[area_name] = process_crome_file(file_path, lookup_df, crome_to_lccm_map)
        print(f"Processed: {area_name}")
    except Exception as e:
        print(f"Error processing {area_name}: {e}")
print("\nAll areas processed.")
first_area = list(processed_gdfs.keys())[0]
print(f"\n--- {first_area} mapped class distribution ---")
display(processed_gdfs[first_area]['lccm_aligned_name'].value_counts())

In [ ]:
#Save aligned GeoJSON files
output_dir = '/content/crome_2022_aligned'
os.makedirs(output_dir, exist_ok=True)
print(f"Saving files to: {output_dir}")
for area_name, gdf in processed_gdfs.items():
    file_name = f"Crop_Map_of_England_2022_{area_name}_aligned.geojson"
    output_path = os.path.join(output_dir, file_name)
    try:
        gdf.to_file(output_path, driver='GeoJSON')
        print(f"Saved: {file_name}")
    except Exception as e:
        print(f"Error saving {area_name}: {e}")
print("\nAll files saved.")

In [ ]:
#Refine class mappings and metadata.
def get_mapping_metadata(crome_name):
    main_crop_map = {
        'Winter Wheat': 'Winter wheat', 'Spring Wheat': 'Spring wheat',
        'Winter Barley': 'Winter barley', 'Spring Barley': 'Spring barley',
        'Winter Oats': 'Winter oats', 'Spring Oats': 'Spring oats',
        'Winter Field beans': 'Winter field beans', 'Spring Field beans': 'Spring field beans',
        'Winter Oilseed': 'Oilseed rape', 'Maize': 'Maize',
        'Potato': 'Potatoes', 'Beet': 'Beet (sugar beet / fodder beet)',
        'Grass': 'Grass', 'Spring Peas': 'Peas'
    }
    other_crop_map = {
        'Clover': 'Other crops', 'Lucerne': 'Other crops', 'Onions': 'Other crops',
        'Spring Cabbage': 'Other crops', 'Spring Linseed': 'Other crops',
        'Winter Linseed': 'Other crops', 'Winter Rye': 'Other crops',
        'Winter Triticale': 'Other crops', 'Mixed Crop-Group 1': 'Other crops'
    }
    exclude_map = {
        'Fallow Land': 'Exclude', 'Water': 'Exclude',
        'Non-vegetated or sparsely-vegetated Land': 'Exclude',
        'Trees and Scrubs, short Woody plants, hedgerows': 'Exclude',
        'Perennial Crops and Isolated Trees': 'Exclude'
    }
    if crome_name in main_crop_map:
        return main_crop_map[crome_name], 'direct', 'retain'
    elif crome_name in other_crop_map:
        return other_crop_map[crome_name], 'aggregate', 'aggregate'
    elif crome_name in exclude_map:
        return exclude_map[crome_name], 'non-crop', 'exclude'
    else:
        return 'Unknown', 'ambiguous', 'exclude'
def process_crome_file_refined(file_path, lookup_df):
    gdf = gpd.read_file(file_path)
    lookup_col = 'lucode' if 'lucode' in lookup_df.columns else 'LUCODE'
    gdf = gdf.merge(lookup_df, left_on='lucode', right_on=lookup_col, how='left')
    desc_col = 'Land Use Description' if 'Land Use Description' in gdf.columns else 'Land Cover Description'
    gdf['crome_original_name'] = gdf[desc_col]
    meta_results = gdf['crome_original_name'].apply(get_mapping_metadata)
    gdf['ukceh_aligned_name'] = meta_results.apply(lambda x: x[0])
    gdf['match_type'] = meta_results.apply(lambda x: x[1])
    gdf['decision'] = meta_results.apply(lambda x: x[2])
    gdf['analysis_class'] = gdf['ukceh_aligned_name']
    return gdf
refined_output_dir = '/content/crome_2022_refined'
os.makedirs(refined_output_dir, exist_ok=True)
for file_path in all_crome_files:
    area_name = os.path.basename(file_path).replace('Crop_Map_of_England_2022_', '').replace('.geojson', '')
    refined_gdf = process_crome_file_refined(file_path, lookup_df)
    output_path = os.path.join(refined_output_dir, f"Crop_Map_of_England_2022_{area_name}_refined.geojson")
    refined_gdf.to_file(output_path, driver='GeoJSON')
    print(f"Processed and saved refined data: {area_name}")
display(refined_gdf[['crome_original_name', 'ukceh_aligned_name', 'match_type', 'decision']].head())

In [ ]:
#Copy refined data to Google Drive.
import shutil
source_dir = '/content/crome_2022_refined'
dest_dir = '/content/drive/MyDrive/Dissertation/crome/2022/crome_2022_refined/east_anglia'
if not os.path.exists(source_dir):
    raise FileNotFoundError(f'Missing source directory: {source_dir}')

os.makedirs(dest_dir, exist_ok=True)

refined_files = sorted(
    glob.glob(os.path.join(source_dir, '*_refined.geojson'))
)

if len(refined_files) != 3:
    raise RuntimeError(
        f'Expected three refined GeoJSON files, found {len(refined_files)}'
    )

for source_path in refined_files:
    destination_path = os.path.join(
        dest_dir,
        os.path.basename(source_path)
    )
    shutil.copy2(source_path, destination_path)
    print(f'Saved: {destination_path}')